Imports

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms

from helper import make_subset, train, test

Use device

In [17]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


Fill in respective folder location for dataset

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train"
valid_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\valid"

train_full = datasets.ImageFolder(train_dir, transform=transform)
valid_full = datasets.ImageFolder(valid_dir, transform=transform)

print(train_full.classes)       # ['fake', 'real']
print(train_full.class_to_idx) # {'fake': 0, 'real': 1}

Train classes: ['fake', 'real']
Train mapping: {'fake': 0, 'real': 1}


In [ ]:
def get_class_indices(dataset, class_name):
    class_idx = dataset.class_to_idx[class_name]
    return [i for i, (_, label) in enumerate(dataset.samples) if label == class_idx]


def make_subset(dataset, n_real, n_fake):
    real_idx = get_class_indices(dataset, "real")
    fake_idx = get_class_indices(dataset, "fake")

    n_real = min(n_real, len(real_idx))
    n_fake = min(n_fake, len(fake_idx))

    real_pick = torch.randperm(len(real_idx))[:n_real].tolist()
    fake_pick = torch.randperm(len(fake_idx))[:n_fake].tolist()

    indices = [real_idx[i] for i in real_pick] + [fake_idx[i] for i in fake_pick]

    # shuffle
    indices = torch.tensor(indices)[torch.randperm(len(indices))].tolist()

    return Subset(dataset, indices)

In [ ]:
train_dataset = make_subset(train_full, 5000, 5000)
valid_dataset = make_subset(valid_full, 1000, 1000)

In [ ]:
val_size = int(0.5 * len(valid_dataset))
test_size = len(valid_dataset) - val_size

val_dataset, test_dataset = random_split(valid_dataset, [val_size, test_size])

Create train, valid and test datasets and dataloaders

In [19]:
# train_dataset = ConcatDataset([
#     make_subset(train_real_dir, label=0, n=9000),
#     make_subset(train_fake_dir, label=1, n=1000)
# ])

# valid_dataset = ConcatDataset([
#     make_subset(val_real_dir, label=0, n=9000),
#     make_subset(val_fake_dir, label=1, n=1000)
# ])

# val_size = int(0.5 * len(valid_dataset))
# test_size = len(valid_dataset) - val_size

# val_dataset, test_dataset = random_split(valid_dataset, [val_size, test_size])

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# real_count = 0
# fake_count = 0

# for _, labels in train_loader:
#     real_count += (labels == 0).sum().item()
#     fake_count += (labels == 1).sum().item()

# print("Train real:", real_count)
# print("Train fake:", fake_count)

CNN model

In [20]:
class AnomalyCNN(nn.Module):
    def __init__(self):
        super(AnomalyCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.AdaptiveAvgPool2d((4,4))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),  # for 256 x 256 input
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.features(x)   # convolution
        x = self.fc(x)
        return x

In [21]:
model = AnomalyCNN().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([9.0]).to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001)
train(model, train_loader, val_loader, criterion, optimizer, device)
test(model, test_loader, device)

Epoch [1/5], Validation Accuracy: 60.75%, Loss: 0.6347
Epoch [2/5], Validation Accuracy: 60.75%, Loss: 0.5716
Epoch [3/5], Validation Accuracy: 60.75%, Loss: 0.5605
Epoch [4/5], Validation Accuracy: 60.75%, Loss: 0.5584
Epoch [5/5], Validation Accuracy: 60.75%, Loss: 0.5537
Test accuracy: 60.75
